# Tutorial 0: Onboarding Roadmap (Notebook-First)

This notebook is the entry point for a 9-part Metamodeler tutorial track.
It is designed for lab members with general programming and biological modeling background.


## Environment first (simple rule)

Use a Jupyter kernel that already belongs to the conda (or venv) environment you
want. That kernel environment is what persists across all notebook cells.

You don't need to memorize environment names — the preflight cell below runs
**`bayesmm doctor`**, which reports your actual OS, Python, environment, and which
optional backends (PyMC / SBI) are installed. If something is missing, run
**`bayesmm setup`** for platform-correct install commands.

No installation commands run automatically in this notebook.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
if not (project_root / "src").is_dir() and (project_root.parent / "src").is_dir():
    project_root = project_root.parent

os.chdir(project_root)
src_path = project_root / "src"
if src_path.is_dir() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

env = os.environ.copy()
src_str = str(src_path)
# os.pathsep is ":" on POSIX and ";" on Windows — never hardcode the separator.
env["PYTHONPATH"] = (
    src_str
    if not env.get("PYTHONPATH")
    else os.pathsep.join([src_str, env["PYTHONPATH"]])
)

print(f"Project root: {project_root}")
print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version.split()[0]}")

# CLI preflight — confirms the package is importable in this kernel.
result = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "--version"],
    cwd=project_root,
    env=env,
    text=True,
    capture_output=True,
)
if result.returncode != 0:
    raise RuntimeError(
        "CLI preflight failed. Make sure this notebook runs in a prepared environment.\n"
        f"stderr:\n{result.stderr}"
    )
print(result.stdout.strip())

# Environment diagnostic — `bayesmm doctor` reports OS, Python, conda/venv, and
# which optional backends (pymc / sbi / torch) are available. This is the fastest
# way to confirm your kernel is ready for the tutorials below; if a backend is
# missing, run `bayesmm setup` for platform-correct install commands.
doctor = subprocess.run(
    [sys.executable, "-m", "bayesian_metamodeling.cli.main", "doctor"],
    cwd=project_root,
    env=env,
    text=True,
    capture_output=True,
)
print()
print(doctor.stdout.strip() or doctor.stderr.strip())


## Optional commands (manual, uncomment if needed)

Use these in a terminal or uncomment in a notebook cell if you explicitly want them.
Adjust environment names to your setup.


In [ ]:
## `bayesmm setup` generates the correct install commands for *your* platform
## (conda vs pip, Windows vs macOS/Linux quoting). Run it instead of guessing:
#   python -m bayesian_metamodeling.cli.main setup
#
## Or install manually in a terminal (adjust to your environment):
## PyMC backend  — learning/coupling with PyMC:
#   conda install -c conda-forge pymc arviz       # or: pip install 'bayesian-metamodeling[pymc]'
## SBI backend   — learning/coupling with SBI:
#   conda install -c conda-forge pytorch sbi      # or: pip install 'bayesian-metamodeling[sbi]'
## BioModels (Tutorial 2) — SBML model execution:
#   pip install libroadrunner tellurium

## Package status
Report what packages are available in current environment

In [9]:
import importlib.util
import importlib.metadata as md

packages = ["pymc", "arviz", "torch", "sbi", "tellurium"]
status = {}
for pkg in packages:
    if importlib.util.find_spec(pkg) is None:
        status[pkg] = "not_installed"
    else:
        try:
            status[pkg] = md.version(pkg)
        except md.PackageNotFoundError:
            status[pkg] = "installed"

for pkg, ver in status.items():
    # print right aligned package name and version
    print(f"{pkg:>20}: {ver}")


# Print libroadrunner version using roadrunner API if available
try:
    import roadrunner
    print(f"{'libroadrunner (API)':>20}: {roadrunner.__version__}")
except ImportError:
    print(f"{'libroadrunner (API)':>20}: not_installed")

                pymc: 5.27.1
               arviz: 0.23.4
               torch: 2.2.2
                 sbi: not_installed
           tellurium: 2.2.11.2
 libroadrunner (API): 2.9.0


## Tutorial map
1. `Tutorial_1.ipynb`: quick toy-model scientific run
2. `Tutorial_2.ipynb`: early BioModels run
3. `Tutorial_3.ipynb`: spec contract debugging
4. `Tutorial_4.ipynb`: DOE strategy comparison
5. `Tutorial_5.ipynb`: PyMC surrogate learning
6. `Tutorial_6.ipynb`: SBI surrogate learning
7. `Tutorial_7.ipynb`: two-model coupling
8. `Tutorial_8.ipynb`: three-model coupling
9. `Tutorial_9.ipynb`: reproducible capstone


In [ ]:
import matplotlib.pyplot as plt

labels = ["T1", "T2", "T3", "T4", "T5", "T6", "T7", "T8", "T9"]
complexity = [1, 2, 2, 3, 4, 4, 5, 6, 7]

plt.figure(figsize=(8, 3))
plt.plot(labels, complexity, marker="o")
plt.title("Tutorial progression (complexity)")
plt.xlabel("Tutorial")
plt.ylabel("Relative complexity")
plt.grid(True, alpha=0.3)
plt.show()


## Tips for new lab members
- Keep a short run log: commands, run IDs, and interpretation notes.
- If a dependency is missing, continue with fallback paths and return later.
- Inspect one artifact file after each major command.
